# EXPLAIN ANTES DE OPTIMIZAR

In [1]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(dbname="bookstore_g3", host="localhost", user="postgres", password="postgres")
conn.autocommit = True

def run(sql):
    """Ejecuta una o más sentencias SQL. Si la última devuelve filas, las
    retorna como DataFrame; si no, no retorna nada."""
    with conn.cursor() as cur:
        cur.execute(sql)
        if cur.description is not None:
            cols = [d.name for d in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)

def explain(sql):
    """Imprime el plan de un EXPLAIN/EXPLAIN ANALYZE línea por línea."""
    df = run(sql)
    print("\n".join(df.iloc[:, 0].tolist()))

## EXPLAIN de TRIPLE join, doble agruptaciones y ordenar (ORDER BY MONTH,REVENUE DESC ).

In [9]:
explain("""
EXPLAIN
SELECT b.genre, DATE_TRUNC('month', o.order_date) AS month,
       SUM(oi.quantity * oi.unit_price) AS revenue
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN books b        ON b.book_id   = oi.book_id
WHERE o.order_date >= '2023-12-01' AND o.order_date < '2024-12-01'
GROUP BY b.genre, DATE_TRUNC('month', o.order_date)
ORDER BY month, revenue DESC;
""")

Incremental Sort  (cost=202437.79..439750.91 rows=1387439 width=47)
  Sort Key: (date_trunc('month'::text, o.order_date)), (sum(((oi.quantity)::numeric * oi.unit_price))) DESC
  Presorted Key: (date_trunc('month'::text, o.order_date))
  ->  GroupAggregate  (cost=202437.39..402182.10 rows=1387439 width=47)
        Group Key: (date_trunc('month'::text, o.order_date)), b.genre
        ->  Gather Merge  (cost=202437.39..364027.53 rows=1387439 width=25)
              Workers Planned: 2
              ->  Sort  (cost=201437.37..202882.62 rows=578100 width=25)
                    Sort Key: (date_trunc('month'::text, o.order_date)), b.genre
                    ->  Hash Join  (cost=40161.16..132278.42 rows=578100 width=25)
                          Hash Cond: (oi.book_id = b.book_id)
                          ->  Parallel Hash Join  (cost=39125.16..128279.45 rows=578100 width=22)
                                Hash Cond: (oi.order_id = o.order_id)
                                ->  Parallel Se

In [ ]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT b.genre, DATE_TRUNC('month', o.order_date) AS month,
       SUM(oi.quantity * oi.unit_price) AS revenue
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN books b        ON b.book_id   = oi.book_id
WHERE o.order_date >= '2023-12-01' AND o.order_date < '2024-12-01'
GROUP BY b.genre, DATE_TRUNC('month', o.order_date)
ORDER BY month, revenue DESC;
""")

Incremental Sort  (cost=202437.79..439750.91 rows=1387439 width=47) (actual time=2106.654..3109.051 rows=180.00 loops=1)
  Sort Key: (date_trunc('month'::text, o.order_date)), (sum(((oi.quantity)::numeric * oi.unit_price))) DESC
  Presorted Key: (date_trunc('month'::text, o.order_date))
  Full-sort Groups: 4  Sort Method: quicksort  Average Memory: 27kB  Peak Memory: 27kB
  Buffers: shared hit=1677 read=52932, temp read=34747 written=34844
  ->  GroupAggregate  (cost=202437.39..402182.10 rows=1387439 width=47) (actual time=1756.317..3108.079 rows=180.00 loops=1)
        Group Key: (date_trunc('month'::text, o.order_date)), b.genre
        Buffers: shared hit=1677 read=52932, temp read=34747 written=34844
        ->  Gather Merge  (cost=202437.39..364027.53 rows=1387439 width=25) (actual time=1746.648..2448.428 rows=1392554.00 loops=1)
              Workers Planned: 2
              Workers Launched: 2
              Buffers: shared hit=1677 read=52932, temp read=34747 written=34844
     

In [3]:
explain("""
EXPLAIN ANALYZE
SELECT b.genre, DATE_TRUNC('month', o.order_date) AS month,
       SUM(oi.quantity * oi.unit_price) AS revenue
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN books b        ON b.book_id   = oi.book_id
WHERE o.order_date >= '2023-12-01' AND o.order_date < '2024-12-01'
GROUP BY b.genre, DATE_TRUNC('month', o.order_date)
ORDER BY month, revenue DESC;
""")

Incremental Sort  (cost=202437.79..439750.91 rows=1387439 width=47) (actual time=2259.224..3291.363 rows=180.00 loops=1)
  Sort Key: (date_trunc('month'::text, o.order_date)), (sum(((oi.quantity)::numeric * oi.unit_price))) DESC
  Presorted Key: (date_trunc('month'::text, o.order_date))
  Full-sort Groups: 4  Sort Method: quicksort  Average Memory: 27kB  Peak Memory: 27kB
  Buffers: shared hit=755 read=53857, temp read=34747 written=34833
  ->  GroupAggregate  (cost=202437.39..402182.10 rows=1387439 width=47) (actual time=1899.387..3290.474 rows=180.00 loops=1)
        Group Key: (date_trunc('month'::text, o.order_date)), b.genre
        Buffers: shared hit=752 read=53857, temp read=34747 written=34833
        ->  Gather Merge  (cost=202437.39..364027.53 rows=1387439 width=25) (actual time=1885.562..2599.449 rows=1392554.00 loops=1)
              Workers Planned: 2
              Workers Launched: 2
              Buffers: shared hit=752 read=53857, temp read=34747 written=34833
        

## EXPLAIN JOIN con triple join, doble agrupación y ordenamiento (ORDER BY REVENUE DESC)

In [8]:
explain("""
EXPLAIN
SELECT b.book_id, b.title, SUM(oi.quantity * oi.unit_price) AS revenue
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN books b        ON b.book_id   = oi.book_id
WHERE o.order_date >= '2023-10-01' AND o.order_date < '2023-11-01'
GROUP BY b.book_id, b.title
ORDER BY revenue DESC LIMIT 10;""")

Limit  (cost=108513.18..108513.21 rows=10 width=52)
  ->  Sort  (cost=108513.18..108588.18 rows=30000 width=52)
        Sort Key: (sum(((oi.quantity)::numeric * oi.unit_price))) DESC
        ->  Finalize HashAggregate  (cost=107489.89..107864.89 rows=30000 width=52)
              Group Key: b.book_id
              ->  Gather  (cost=99374.89..106949.89 rows=72000 width=52)
                    Workers Planned: 2
                    ->  Partial HashAggregate  (cost=98374.89..98749.89 rows=30000 width=52)
                          Group Key: b.book_id
                          ->  Hash Join  (cost=35890.57..97909.14 rows=46575 width=30)
                                Hash Cond: (oi.book_id = b.book_id)
                                ->  Parallel Hash Join  (cost=34854.57..96750.86 rows=46575 width=14)
                                      Hash Cond: (oi.order_id = o.order_id)
                                      ->  Parallel Seq Scan on order_items oi  (cost=0.00..56076.10 rows=2217210 

In [10]:
explain("""
EXPLAIN ANALYZE
SELECT b.book_id, b.title, SUM(oi.quantity * oi.unit_price) AS revenue
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN books b        ON b.book_id   = oi.book_id
WHERE o.order_date >= '2023-10-01' AND o.order_date < '2023-11-01'
GROUP BY b.book_id, b.title
ORDER BY revenue DESC LIMIT 10;""")

Limit  (cost=108513.18..108513.21 rows=10 width=52) (actual time=555.445..556.607 rows=10.00 loops=1)
  Buffers: shared hit=3339 read=51240, temp read=102 written=266
  ->  Sort  (cost=108513.18..108588.18 rows=30000 width=52) (actual time=555.442..556.602 rows=10.00 loops=1)
        Sort Key: (sum(((oi.quantity)::numeric * oi.unit_price))) DESC
        Sort Method: top-N heapsort  Memory: 26kB
        Buffers: shared hit=3339 read=51240, temp read=102 written=266
        ->  Finalize HashAggregate  (cost=107489.89..107864.89 rows=30000 width=52) (actual time=522.791..547.142 rows=26205.00 loops=1)
              Group Key: b.book_id
              Batches: 5  Memory Usage: 8497kB  Disk Usage: 1544kB
              Buffers: shared hit=3339 read=51240, temp read=102 written=266
              ->  Gather  (cost=99374.89..106949.89 rows=72000 width=52) (actual time=445.501..466.462 rows=43955.00 loops=1)
                    Workers Planned: 2
                    Workers Launched: 2
          

In [11]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT b.book_id, b.title, SUM(oi.quantity * oi.unit_price) AS revenue
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN books b        ON b.book_id   = oi.book_id
WHERE o.order_date >= '2023-10-01' AND o.order_date < '2023-11-01'
GROUP BY b.book_id, b.title
ORDER BY revenue DESC LIMIT 10;""")

Limit  (cost=108513.18..108513.21 rows=10 width=52) (actual time=561.773..563.039 rows=10.00 loops=1)
  Buffers: shared hit=3903 read=50676, temp read=103 written=266
  ->  Sort  (cost=108513.18..108588.18 rows=30000 width=52) (actual time=561.770..563.032 rows=10.00 loops=1)
        Sort Key: (sum(((oi.quantity)::numeric * oi.unit_price))) DESC
        Sort Method: top-N heapsort  Memory: 26kB
        Buffers: shared hit=3903 read=50676, temp read=103 written=266
        ->  Finalize HashAggregate  (cost=107489.89..107864.89 rows=30000 width=52) (actual time=526.730..552.552 rows=26205.00 loops=1)
              Group Key: b.book_id
              Batches: 5  Memory Usage: 8497kB  Disk Usage: 1544kB
              Buffers: shared hit=3903 read=50676, temp read=103 written=266
              ->  Gather  (cost=99374.89..106949.89 rows=72000 width=52) (actual time=447.891..470.100 rows=43969.00 loops=1)
                    Workers Planned: 2
                    Workers Launched: 2
          

## EXPLAIN Consultas REVIEW

In [ ]:
explain("""
EXPLAIN
SELECT review_id, user_id, rating, review_text, review_date
FROM reviews
WHERE book_id = 20053
ORDER BY review_date DESC LIMIT 20;""")


Limit  (cost=19161.42..19163.75 rows=20 width=84)
  ->  Gather Merge  (cost=19161.42..19172.49 rows=95 width=84)
        Workers Planned: 2
        ->  Sort  (cost=18161.40..18161.50 rows=40 width=84)
              Sort Key: review_date DESC
              ->  Parallel Seq Scan on reviews  (cost=0.00..18160.33 rows=40 width=84)
                    Filter: (book_id = 20053)


In [15]:
explain("""
EXPLAIN ANALYZE
SELECT review_id, user_id, rating, review_text, review_date
FROM reviews
WHERE book_id = 20053
ORDER BY review_date DESC LIMIT 20;""")

Limit  (cost=19161.42..19163.75 rows=20 width=84) (actual time=95.213..98.845 rows=20.00 loops=1)
  Buffers: shared hit=638 read=12388
  ->  Gather Merge  (cost=19161.42..19172.49 rows=95 width=84) (actual time=95.212..98.841 rows=20.00 loops=1)
        Workers Planned: 2
        Workers Launched: 2
        Buffers: shared hit=638 read=12388
        ->  Sort  (cost=18161.40..18161.50 rows=40 width=84) (actual time=68.121..68.122 rows=17.33 loops=3)
              Sort Key: review_date DESC
              Sort Method: top-N heapsort  Memory: 28kB
              Buffers: shared hit=638 read=12388
              Worker 0:  Sort Method: top-N heapsort  Memory: 28kB
              Worker 1:  Sort Method: top-N heapsort  Memory: 28kB
              ->  Parallel Seq Scan on reviews  (cost=0.00..18160.33 rows=40 width=84) (actual time=0.670..67.740 rows=598.00 loops=3)
                    Filter: (book_id = 20053)
                    Rows Removed by Filter: 332735
                    Buffers: shared

In [16]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT review_id, user_id, rating, review_text, review_date
FROM reviews
WHERE book_id = 20053
ORDER BY review_date DESC LIMIT 20;""")

Limit  (cost=19161.42..19163.75 rows=20 width=84) (actual time=86.186..90.801 rows=20.00 loops=1)
  Buffers: shared hit=920 read=12106
  ->  Gather Merge  (cost=19161.42..19172.49 rows=95 width=84) (actual time=86.183..90.793 rows=20.00 loops=1)
        Workers Planned: 2
        Workers Launched: 2
        Buffers: shared hit=920 read=12106
        ->  Sort  (cost=18161.40..18161.50 rows=40 width=84) (actual time=57.960..57.963 rows=18.00 loops=3)
              Sort Key: review_date DESC
              Sort Method: top-N heapsort  Memory: 28kB
              Buffers: shared hit=920 read=12106
              Worker 0:  Sort Method: top-N heapsort  Memory: 29kB
              Worker 1:  Sort Method: top-N heapsort  Memory: 28kB
              ->  Parallel Seq Scan on reviews  (cost=0.00..18160.33 rows=40 width=84) (actual time=1.279..57.619 rows=598.00 loops=3)
                    Filter: (book_id = 20053)
                    Rows Removed by Filter: 332735
                    Buffers: shared

## EXPLAIN de una busqueda simple con función LOWER

In [ ]:
explain(""" 
EXPLAIN
SELECT user_id, is_premium
FROM users
WHERE LOWER(email) = LOWER('ROSA.FERREIRA47314@SHOP.NET');""")

Gather  (cost=1000.00..4664.71 rows=1000 width=5)
  Workers Planned: 1
  ->  Parallel Seq Scan on users  (cost=0.00..3564.71 rows=588 width=5)
        Filter: (lower(email) = 'rosa.ferreira47314@shop.net'::text)


In [18]:
explain(""" 
EXPLAIN ANALYZE
SELECT user_id, is_premium
FROM users
WHERE LOWER(email) = LOWER('ROSA.FERREIRA47314@SHOP.NET');""")

Gather  (cost=1000.00..4664.71 rows=1000 width=5) (actual time=30.700..68.671 rows=1.00 loops=1)
  Workers Planned: 1
  Workers Launched: 1
  Buffers: shared read=1800
  ->  Parallel Seq Scan on users  (cost=0.00..3564.71 rows=588 width=5) (actual time=28.174..45.790 rows=0.50 loops=2)
        Filter: (lower(email) = 'rosa.ferreira47314@shop.net'::text)
        Rows Removed by Filter: 100000
        Buffers: shared read=1800
Planning Time: 0.071 ms
Execution Time: 68.706 ms


In [2]:
explain(""" 
EXPLAIN (ANALYZE, BUFFERS)
SELECT user_id, is_premium, country
FROM users
WHERE email = 'Zara.moreno33740@mail.com';""")

Index Scan using users_email_key on users  (cost=0.42..8.44 rows=1 width=8) (actual time=0.935..0.936 rows=1.00 loops=1)
  Index Cond: (email = 'Zara.moreno33740@mail.com'::text)
  Index Searches: 1
  Buffers: shared read=4
Planning:
  Buffers: shared hit=62 read=24
Planning Time: 9.964 ms
Execution Time: 1.051 ms


## EXPLAIN Consultas que ocupan order_date para filtrar

In [20]:
explain(""" 
EXPLAIN
SELECT o.order_id, o.user_id, o.order_date, o.total_amount
FROM orders o
JOIN users u ON u.user_id = o.user_id
WHERE u.is_premium = TRUE
AND o.order_date >= '2024-12-15' AND o.order_date < '2024-12-22'
ORDER BY o.order_date DESC;""")

Gather Merge  (cost=38798.86..39099.93 rows=2585 width=22)
  Workers Planned: 2
  ->  Sort  (cost=37798.84..37801.53 rows=1077 width=22)
        Sort Key: o.order_date DESC
        ->  Parallel Hash Join  (cost=3124.51..37744.60 rows=1077 width=22)
              Hash Cond: (o.user_id = u.user_id)
              ->  Parallel Seq Scan on orders o  (cost=0.00..34592.00 rows=10700 width=22)
                    Filter: ((order_date >= '2024-12-15 00:00:00'::timestamp without time zone) AND (order_date < '2024-12-22 00:00:00'::timestamp without time zone))
              ->  Parallel Hash  (cost=2976.47..2976.47 rows=11843 width=4)
                    ->  Parallel Seq Scan on users u  (cost=0.00..2976.47 rows=11843 width=4)
                          Filter: is_premium


In [21]:
explain(""" 
EXPLAIN ANALYZE
SELECT o.order_id, o.user_id, o.order_date, o.total_amount
FROM orders o
JOIN users u ON u.user_id = o.user_id
WHERE u.is_premium = TRUE
AND o.order_date >= '2024-12-15' AND o.order_date < '2024-12-22'
ORDER BY o.order_date DESC;""")

Gather Merge  (cost=38798.86..39099.93 rows=2585 width=22) (actual time=150.881..155.207 rows=2434.00 loops=1)
  Workers Planned: 2
  Workers Launched: 2
  Buffers: shared hit=3506 read=17900
  ->  Sort  (cost=37798.84..37801.53 rows=1077 width=22) (actual time=125.044..125.096 rows=811.33 loops=3)
        Sort Key: o.order_date DESC
        Sort Method: quicksort  Memory: 61kB
        Buffers: shared hit=3506 read=17900
        Worker 0:  Sort Method: quicksort  Memory: 52kB
        Worker 1:  Sort Method: quicksort  Memory: 55kB
        ->  Parallel Hash Join  (cost=3124.51..37744.60 rows=1077 width=22) (actual time=5.372..124.534 rows=811.33 loops=3)
              Hash Cond: (o.user_id = u.user_id)
              Buffers: shared hit=3492 read=17900
              ->  Parallel Seq Scan on orders o  (cost=0.00..34592.00 rows=10700 width=22) (actual time=0.388..116.606 rows=8107.33 loops=3)
                    Filter: ((order_date >= '2024-12-15 00:00:00'::timestamp without time zone) AN

In [22]:
explain(""" 
EXPLAIN (ANALYZE, BUFFERS)
SELECT o.order_id, o.user_id, o.order_date, o.total_amount
FROM orders o
JOIN users u ON u.user_id = o.user_id
WHERE u.is_premium = TRUE
AND o.order_date >= '2024-12-15' AND o.order_date < '2024-12-22'
ORDER BY o.order_date DESC;""")

Gather Merge  (cost=38798.86..39099.93 rows=2585 width=22) (actual time=141.025..145.629 rows=2434.00 loops=1)
  Workers Planned: 2
  Workers Launched: 2
  Buffers: shared hit=3788 read=17618
  ->  Sort  (cost=37798.84..37801.53 rows=1077 width=22) (actual time=116.450..116.492 rows=811.33 loops=3)
        Sort Key: o.order_date DESC
        Sort Method: quicksort  Memory: 61kB
        Buffers: shared hit=3788 read=17618
        Worker 0:  Sort Method: quicksort  Memory: 54kB
        Worker 1:  Sort Method: quicksort  Memory: 54kB
        ->  Parallel Hash Join  (cost=3124.51..37744.60 rows=1077 width=22) (actual time=6.138..115.999 rows=811.33 loops=3)
              Hash Cond: (o.user_id = u.user_id)
              Buffers: shared hit=3774 read=17618
              ->  Parallel Seq Scan on orders o  (cost=0.00..34592.00 rows=10700 width=22) (actual time=0.809..108.195 rows=8107.33 loops=3)
                    Filter: ((order_date >= '2024-12-15 00:00:00'::timestamp without time zone) AN